# Complejidad, sesgo y varianza

Aprender curvas y entender lo que cuestan

## La representación decide qué se puede aprender

El capítulo anterior terminó con un diagnóstico: la clase de hipótesis considerada hasta ahora (funciones lineales) puede ser insuficiente.

Conviene ver primero qué tiene de limitante esa suposición. Para ello, simularemos una señal **deliberadamente** no lineal.

In [ ]:
import numpy as np
import torch
from matplotlib import pyplot as plt

torch.set_default_dtype(torch.float64)

AZUL, GRIS, NARANJA, TINTA = "#151f6c", "#8b8e95", "#ff5700", "#1b1d21"
NOISE = 0.2
REPETICIONES = 2000      
rejilla = torch.linspace(0, 2, 400)                         


def senal(x):
    return torch.sin(torch.pi * x)


def genera(n, semilla):
    torch.manual_seed(semilla)
    x = torch.rand(n) * 2
    return x, senal(x) + NOISE * torch.randn(n)


x, y = genera(36, semilla=42)
print(f"{len(x)} observaciones, entrada en [0, 2], ruido de nivel {NOISE}")
print(f"suelo de ruido: sigma^2 = {NOISE ** 2:.4f}")

La señal es $f(x)=\sin(\pi x)$ sobre $[0,2]$. El ruido es gaussiano de nivel $\sigma=0.2$, de modo que ningún modelo puede tener un riesgo menor que $\sigma^2=0.04$.

Figura 1: Las treinta y seis observaciones y la señal que las genera. La señal sube en la primera mitad del recorrido y baja en la segunda, así que ninguna recta puede seguirla: lo que una recta gane en un tramo lo pierde en el otro.

In [ ]:
fig, ax = plt.subplots(figsize=(6.0, 3.2))
ax.plot(rejilla, senal(rejilla), color=TINTA, lw=1.8, label="señal")
ax.scatter(x.numpy(), y.numpy(), s=18, color=GRIS, alpha=0.75, zorder=3,
           label="las 36 observaciones")
ax.set(xlabel="$x$", ylabel="$y$")
ax.legend(fontsize=8.5, loc="lower left")
plt.tight_layout()

Ahora ajustemos una recta, y veamos qué sucede. Utilizaremos un tamaño muestral creciente. Nótese que, al conocer la señal real, podemos calcular estimar el riesgo verdadero.

In [ ]:
# TODO: completar en clase

El riesgo baja de $0.2809$ a $0.2364$ y ahí se queda. Aumentar el tamaño del conjunto de entrenamiento no es capaz de reducir este riesgo.

Figura 2: La recta ajustada con treinta observaciones y con cien veces más.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8, 3.0), sharey=True)
for panel, n in enumerate([30, 3000]):
    xn, yn = genera(n, semilla=42)
    w = ajusta_recta(xn, yn)
    ax[panel].scatter(xn.numpy(), yn.numpy(), s=8, color=GRIS, alpha=0.35,
                      zorder=2)
    ax[panel].plot(rejilla, senal(rejilla), color=TINTA, lw=1.8,
                   label="señal")
    ax[panel].plot(rejilla, w[0] + w[1] * rejilla, color=NARANJA, lw=1.8,
                   label="recta ajustada")
    ax[panel].set(xlabel="$x$", title=f"n = {n}, riesgo "
                                      f"{riesgo_verdadero_recta(w):.3f}")
ax[0].set(ylabel="$y$")
ax[0].legend(fontsize=8.5, loc="lower left")
plt.tight_layout()

Lo que necesitamos para reducir este riesgo es una clase de hipótesis que contenga funciones no lineales. Hay una forma de conseguirla esto sin abandonar nada de lo que sabemos.

<span class="theorem-title">**Definición 1 (Mapa de características)**</span> Un **mapa de características** es una función $\boldsymbol{\varphi}:\mathbb{R}^{p}\to\mathbb{R}^{d}$ que transforma el vector de variables predictoras en un vector de $d$ **características** $\boldsymbol{\varphi}(\mathbf{x})=\bigl(\varphi_1(\mathbf{x}),\ldots,\varphi_{d}(\mathbf{x})\bigr)^{\mathsf{T}}$. El modelo asociado es

$$
f_{\boldsymbol{w}}(\mathbf{x})=\boldsymbol{w}^{\mathsf{T}}\boldsymbol{\varphi}(\mathbf{x})=\sum_{j=1}^{d}w_j\,\varphi_j(\mathbf{x}),
 \qquad(1)$$

y la **matriz de características** $\boldsymbol{\Phi}\in\mathbb{R}^{n\times d}$ tiene por fila $i$ el vector $\boldsymbol{\varphi}(\mathbf{x}_i)^{\mathsf{T}}$.

Considerando este mapa de características, el modelo generador completo es el gaussiano de **?@def-modelo-gaussiano**, con <a href="#eq-modelo-caracteristicas" class="quarto-xref">Ecuación 1</a> en el lugar de la señal,

$$
y_i\,\vert\,\mathbf{x}_i\sim\mathcal{N}\bigl(\boldsymbol{w}^{\mathsf{T}}\boldsymbol{\varphi}(\mathbf{x}_i),\sigma^2\bigr),
 \qquad(2)$$

que es el modelo lineal-gaussiano de **?@def-lineal-gaussiano** con $\boldsymbol{\varphi}(\mathbf{x}_i)$ en lugar de $\mathbf{x}_i$. **?@cor-mle-minimos-cuadrados** sigue aplicando, y ajustar por máxima verosimilitud sigue siendo minimizar el riesgo cuadrático.

Es importante señalar que la expresión de <a href="#eq-modelo-caracteristicas" class="quarto-xref">Ecuación 1</a> es **no lineal en la entrada** pero **lineal en los parámetros**, esto garantiza lo siguiente.

<span class="theorem-title">**Proposición 1 (Lo visto en el capítulo 3 sigue aplicando)**</span> El problema de minimizar el riesgo cuadrático empírico sobre el modelo <a href="#eq-modelo-caracteristicas" class="quarto-xref">Ecuación 1</a> sobre $\boldsymbol{w}$ es el mismo problema del capítulo 3 con $\boldsymbol{\Phi}$ en lugar de $\mathbf{X}$. En particular, el gradiente es $\frac{2}{n}\boldsymbol{\Phi}^{\mathsf{T}}(\boldsymbol{\Phi}\boldsymbol{w}-\mathbf{y})$, las ecuaciones normales son $\boldsymbol{\Phi}^{\mathsf{T}}\boldsymbol{\Phi}\boldsymbol{w}=\boldsymbol{\Phi}^{\mathsf{T}}\mathbf{y}$, y el ajuste es único si $\boldsymbol{\Phi}$ tiene rango completo por columnas.

<span class="proof-title">*Demostración*. </span>El riesgo empírico es $\hat{R}(\boldsymbol{w})=\frac{1}{n}\left\lVert \mathbf{y}-\boldsymbol{\Phi}\boldsymbol{w} \right\rVert^2$, que es literalmente la expresión de **?@prp-riesgo-matricial** con $\boldsymbol{\Phi}$ en el lugar de $\mathbf{X}$. Ninguna de las demostraciones del capítulo 3 usa de dónde salen las columnas de la matriz, solo que sean números fijos dada la muestra, y las características lo son porque $\boldsymbol{\varphi}$ no depende de $\boldsymbol{w}$. Por tanto se aplican sin cambios **?@thm-gradiente-vectorial** y **?@thm-existencia-unicidad**.

La consecuencia práctica es que ajustar este nuevo modelo no cuesta ni un algoritmo nuevo ni una teoría nueva: simplemente tenemos que construir la nueva matriz de características.

Comencemos por entender un caso sencillo. Si contamos con una única variable predictora, podemos construir un mapa de características tomando sus potencias hasta un grado $d$,

$$
\boldsymbol{\varphi}(x)=(1,x,x^2,\ldots,x^{d})^{\mathsf{T}},
 \qquad(3)$$

de modo que la fila $i$ de $\boldsymbol{\Phi}$ es $(1,x_i,x_i^2,\ldots,x_i^{d})$ y el modelo de <a href="#eq-modelo-caracteristicas" class="quarto-xref">Ecuación 1</a> es el polinomio

$$
f_{\boldsymbol{w}}(x)=w_0+w_1x+w_2x^2+\cdots+w_dx^{d}.
$$

Qué grado $d$ elegir es el asunto que discute este capítulo. Probemos con tres valores diferentes y observemos qué sucede.

In [ ]:
# TODO: completar en clase

Como podemos observar, el riesgo en train baja siempre al añadir grado. No obstante, el riesgo verdadero no: baja de $0.2384$ a $0.0546$ y vuelve a subir a $0.0793$. El modelo de grado 1 no es suficientemente **expresivo** mientras que el modelo de grado 9 está **sobreajustado** (recuerda la **?@def-sobreajuste**). El resto de este capítulo trata de arrojar luz sobre estos conceptos.

Figura 3: Los tres ajustes de la tabla sobre las mismas treinta y seis observaciones. La recta no llega a curvarse; el grado 3 sigue aproximadamente la señal; el grado 9 pasa más cerca de los puntos y por eso baja el riesgo en train.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(9, 2.8), sharey=True)
for panel, grado in enumerate([1, 3, 9]):
    w = ajusta(x, y, grado)
    ax[panel].scatter(x.numpy(), y.numpy(), s=14, color=GRIS, alpha=0.6,
                      zorder=2)
    ax[panel].plot(rejilla, senal(rejilla), color=TINTA, lw=1.5,
                   label="señal")
    ax[panel].plot(rejilla, caracteristicas(rejilla, grado) @ w,
                   color=NARANJA, lw=1.8, zorder=3, label="ajuste")
    ax[panel].set(xlabel="$x$", ylim=(-1.8, 1.8),
                  title=f"grado {grado}, riesgo "
                        f"{riesgo_verdadero(w, grado):.3f}")
ax[0].set(ylabel="$y$")
ax[0].legend(fontsize=8.5, loc="lower left")
plt.tight_layout()

## Características polinómicas e interacciones

Con una sola variable predictora, las características polinómicas son las potencias. Con $p$ variables hay que decidir además qué productos entre ellas se incluyen.

In [ ]:
# TODO: completar en clase

Con las ocho variables numéricas del capítulo 5, pasar a grado 2 son 44 características y a grado 3 son 164. La dimensión crece deprisa, y cada característica nueva es un parámetro más que hay que estimar con las mismas observaciones.

Entre esos productos hay uno de especial relevancia. Un término $x_1x_2$ hace que el efecto de $x_1$ sobre la predicción dependa del valor de $x_2$, porque la derivada de $w_0+w_1x_1+w_2x_2+w_3x_1x_2$ respecto de $x_1$ vale $w_1+w_3x_2$. Ese producto es una **interacción**: sin él, el efecto de $x_1$ sobre la predicción es el mismo cualquiera que sea el valor de $x_2$. Lo observamos en la siguiente figura.

Figura 4: La predicción de $w_0+w_1x_1+w_2x_2+w_3x_1x_2$ frente a $x_1$, para dos valores de $x_2$. Sin el producto las dos rectas son paralelas. Con él las pendientes son $w_1$ y $w_1+w_3$.

In [ ]:
x1 = torch.linspace(0, 1, 100)
casos = [("sin interacción, $w_3 = 0$", 0.0),
         ("con interacción, $w_3 = 1.6$", 1.6)]

fig, ax = plt.subplots(1, 2, figsize=(8, 3.0), sharey=True)
for panel, (titulo, w3) in enumerate(casos):
    for x2, color in [(0.0, AZUL), (1.0, NARANJA)]:
        ax[panel].plot(x1, 0.5 + x1 + 0.4 * x2 + w3 * x1 * x2,
                       color=color, lw=1.8, label=f"$x_2 = {x2:.0f}$")
    ax[panel].set(xlabel="$x_1$", title=titulo)
ax[0].set(ylabel="predicción")
ax[0].legend(fontsize=8.5, loc="upper left")
plt.tight_layout()

## Un algoritmo produce modelos distintos con muestras distintas

Hasta ahora hemos presentado una forma de construir una nueva clase de hipótesis más flexible. No obstante, hemos observado que el riesgo verdadero puede aumentar si nos excedemos con la complejidad (por ejemplo, subiendo demasiado el grado del polinomio).

Para entender por qué ocurre esto, no basta con mirar un único ajuste. Tenemos que evaluar el procedimiento de aprendizaje en su conjunto, estudiando qué sucede si aplicamos nuestro algoritmo a múltiples muestras de entrenamiento distintas extraídas del mismo universo.

En el siguiente experimento vamos a simular 2000 realidades alternativas. En cada una, extraeremos una muestra aleatoria independiente de 36 observaciones, ajustaremos un modelo polinómico y guardaremos la curva resultante. Haremos esto para los grados 1, 3 y 9.

In [ ]:
# TODO: completar en clase

Figura 5: Cien de los dos mil ajustes obtenidos con muestras distintas de treinta y seis observaciones, para tres grados. Con grado 1 las rectas casi coinciden entre sí, pero su promedio no se parece a la señal. Con grado 9 el modelo medio sí calca la señal, pero cada curva individual es errática.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(9, 2.8), sharey=True)
for panel, grado in enumerate([1, 3, 9]):
    P = muchos_ajustes(grado)
    for r in range(0, REPETICIONES, REPETICIONES // 100):
        ax[panel].plot(rejilla, P[r], color=NARANJA, lw=0.6, alpha=0.15)
    ax[panel].plot(rejilla, P.mean(0), color=NARANJA, lw=2.0,
                   label="modelo medio")
    ax[panel].plot(rejilla, senal(rejilla), color=TINTA, lw=1.5,
                   label="señal")
    ax[panel].set(xlabel="$x$", title=f"grado {grado}", ylim=(-2.2, 2.2))
ax[0].set(ylabel="$y$")
ax[0].legend(fontsize=8.5, loc="lower left")
plt.tight_layout()

En la figura se ven tres comportamientos marcadamente diferentes.

- Panel izquierdo (Grado 1): Modelo estable pero sistemáticamente equivocado. El modelo es tan rígido que apenas cambia de una muestra a otra; todas las líneas naranjas están muy juntas (baja varianza). Sin embargo, el modelo medio es incapaz de seguir la curvatura de la señal negra real (alto sesgo).

- Panel derecho (Grado 9): Modelo inestable pero centrado. El modelo es tan flexible que persigue el ruido específico de cada muestra de 36 puntos, produciendo curvas que se disparan en todas direcciones (alta varianza). No obstante, si promediamos esas miles de curvas erráticas, el modelo medio calca la señal subyacente casi a la perfección (bajo sesgo).

- El grado 3 (centro) queda entre los dos extremos: es lo bastante flexible para seguir la curva (bajo sesgo), pero lo bastante rígido para no seguir el ruido de cada muestra (baja varianza).

A continuación, formalizamos matemáticamente este fenómeno.

## La descomposición sesgo-varianza

Queremos responder a la siguiente pregunta: si aplicáramos un mismo procedimiento de ajuste (i.e. algoritmo y clase de hipótesis elegida) repetidas veces sobre distintas muestras de datos de un tamaño dado, ¿cuánto se equivocarían en promedio los modelos resultantes al predecir observaciones futuras?

Para formalizar esto, debemos separar el problema en dos niveles. Primero, fijaremos nuestra atención en una sola observación nueva con características $\mathbf{x}$ concretas (por ejemplo, $x=1.5$). El error de predicción que cometeremos sobre esta observación depende de dos fuentes de aleatoriedad independientes:

1.  La procedente de la muestra de entrenamiento ($\mathcal{D}$): Nuestro modelo ajustado $\hat{f}_{\mathcal{D}}$ es una variable aleatoria, pues depende de la muestra $\mathcal{D}$ (que son $n$ extracciones independientes de la distribución $P^\star$ de **?@def-riesgo-verdadero**) que nos haya tocado para entrenar. Por tanto, calcular el rendimiento promedio exige tomar la esperanza sobre todas las posibles muestras de entrenamiento. Escribimos este valor esperado como $\mathbb{E}_{\mathcal{D}}\!\left[ \cdot \right]$. Con ella se define el **modelo medio**.

$$
\bar{\hat{f}}(\mathbf{x})=\mathbb{E}_{\mathcal{D}}\!\left[ \hat{f}_{\mathcal{D}}(\mathbf{x}) \right] .
 \qquad(4)$$

El modelo medio es, punto a punto, la predicción que da el algoritmo en promedio sobre todas las muestras posibles de tamaño $n$. No es un modelo que se pueda ajustar con unos datos concretos, porque para calcularlo hacen falta todas las muestras posibles: es la curva naranja gruesa de <a href="#fig-variabilidad" class="quarto-xref">Figura 5</a>, el promedio de las dos mil finas.

1.  El ruido del dato futuro ($\varepsilon$): El valor real $Y$ que intentamos predecir no es determinista, pues asumimos que el modelo generador de **?@def-modelo-gaussiano** lo produce como $Y = f(\mathbf{x}) + \varepsilon$. Este ruido es estadísticamente independiente de la muestra de entrenamiento $\mathcal{D}$. De forma análoga al punto anterior, utilizaremos el operador $\mathbb{E}_{\varepsilon}\!\left[ \cdot \right]$ para denotar la esperanza sobre la distribución de este nuevo ruido, del cual asumimos que tiene media cero ($\mathbb{E}_{\varepsilon}\!\left[ \varepsilon \right]=0$) y varianza constante ($\mathrm{Var}\!\left( \varepsilon \right)=\sigma^2$).

Para dar respuesta a la pregunta que abría la sección (cuánto se equivocaría en promedio nuestro procedimiento al predecir), debemos evaluar el error en este punto $\mathbf{x}$ promediando sobre ambas fuentes de aleatoriedad. Matemáticamente, esta métrica es la esperanza conjunta $\mathbb{E}_{\mathcal{D},\varepsilon}\!\left[ (Y - \hat{f}_{\mathcal{D}}(\mathbf{x}))^2 \right]$. El teorema siguiente descompone esta cantidad en tres términos. Para que la identidad se cumpla, hacen falta cuatro supuestos:

1.  Pérdida cuadrática. La descomposición es una propiedad algebraica exclusiva del error cuadrático; con otra función de pérdida no existe una descomposición análoga.

2.  Muestra aleatoria y ajuste determinista. $\mathcal{D}$ está formada por $n$ extracciones independientes de $P^\star$, y el procedimiento de ajuste es una función determinista de $\mathcal{D}$: toda la aleatoriedad de nuestro modelo proviene exclusivamente de la muestra.

3.  Observación futura independiente. El par $(\mathbf{x},Y)$ que se predice no pertenece a $\mathcal{D}$, y su ruido intrínseco $\varepsilon$ es independiente de la muestra de entrenamiento y tiene media cero.

4.  Varianza del ruido constante. $\mathrm{Var}\!\left( \varepsilon \right)=\sigma^2$ no depende de $\mathbf{x}$. Sin esto, al promediar posteriormente sobre todo el espacio de características $X$, el término del ruido no sería un único número $\sigma^2$.

<span class="theorem-title">**Teorema 1 (Descomposición sesgo-varianza)**</span> Fijado un punto $\mathbf{x}$, sea $Y = f(\mathbf{x}) + \varepsilon$ la observación futura real. Sea $\hat{f}_{\mathcal{D}}(\mathbf{x})$ la predicción de nuestro algoritmo entrenado con la muestra aleatoria $\mathcal{D}$, y sea $\bar{\hat{f}}(\mathbf{x})$ el modelo medio de <a href="#eq-modelo-medio" class="quarto-xref">Ecuación 4</a>\].

Bajo los cuatro supuestos anteriores, el error cuadrático esperado se descompone en:

$$
\mathbb{E}_{\mathcal{D},\varepsilon}\!\left[ (Y-\hat{f}_{\mathcal{D}}(\mathbf{x}))^2 \right] =\underbrace{\sigma^2}_{\text{ruido}} +\underbrace{\bigl(f(\mathbf{x}) - \bar{\hat{f}}(\mathbf{x})\bigr)^2}_{\text{sesgo}^2} +\underbrace{\mathbb{E}_{\mathcal{D}}\!\left[ (\hat{f}_{\mathcal{D}}(\mathbf{x})-\bar{\hat{f}}(\mathbf{x}))^2 \right]}_{\text{varianza}} .
 \qquad(5)$$

<span class="proof-title">*Demostración*. </span>Para aligerar la notación, escribiremos $\hat{f}=\hat{f}_{\mathcal{D}}(\mathbf{x})$, $f=f(\mathbf{x})$ y $\bar{\hat{f}}=\bar{\hat{f}}(\mathbf{x})$. La demostración comienza expandiendo el error; para ello, sustituimos $Y$ por su definición ($f+ \varepsilon$) y sumamos y restamos el modelo medio $\bar{\hat{f}}$:

$$
Y-\hat{f}= (f+ \varepsilon) - \hat{f}= \varepsilon + (f-\bar{\hat{f}}) + (\bar{\hat{f}}-\hat{f})
$$

Ahora elevamos esta suma de tres elementos al cuadrado. Obtendremos los tres cuadrados individuales y tres términos cruzados:

$$
(Y-\hat{f})^2 = \varepsilon^2 + (f-\bar{\hat{f}})^2 + (\bar{\hat{f}}-\hat{f})^2 + 2\varepsilon(f-\bar{\hat{f}}) + 2\varepsilon(\bar{\hat{f}}-\hat{f}) + 2(f-\bar{\hat{f}})(\bar{\hat{f}}-\hat{f})
$$

Al aplicar la esperanza conjunta $\mathbb{E}_{\mathcal{D},\varepsilon}\!\left[ \cdot \right]$, la linealidad de la esperanza nos permite analizar cada componente por separado. Veamos paso a paso cómo los tres términos cruzados se anulan:

1.  **El cruce entre el ruido y el sesgo:** $$ 
    \mathbb{E}_{\mathcal{D},\varepsilon}\!\left[ 2\varepsilon(f-\bar{\hat{f}}) \right] = 2(f-\bar{\hat{f}})\,\mathbb{E}_{\varepsilon}\!\left[ \varepsilon \right] = 0 
    $$ Una vez fijado $\mathbf{x}$, tanto la señal verdadera $f$ como el modelo medio $\bar{\hat{f}}$ son constantes (no dependen de la muestra concreta ni del ruido). Por tanto, salen fuera de la esperanza. Como asumimos que el ruido tiene media cero ($\mathbb{E}_{\varepsilon}\!\left[ \varepsilon \right]=0$), todo el término se anula.

2.  **El cruce entre el ruido y la varianza:** $$ 
    \mathbb{E}_{\mathcal{D},\varepsilon}\!\left[ 2\varepsilon(\bar{\hat{f}}-\hat{f}) \right] = 2\,\mathbb{E}_{\varepsilon}\!\left[ \varepsilon \right]\,\mathbb{E}_{\mathcal{D}}\!\left[ \bar{\hat{f}}-\hat{f} \right] = 0 
    $$ Aquí resulta vital el tercer supuesto. Como el ruido de la observación futura ($\varepsilon$) es estadísticamente independiente de la muestra de entrenamiento ($\mathcal{D}$), la esperanza conjunta de su producto se separa en el producto de sus esperanzas individuales. Al ser $\mathbb{E}_{\varepsilon}\!\left[ \varepsilon \right]=0$, el término entero desaparece.

3.  **El cruce entre el sesgo y la varianza:** $$ \mathbb{E}_{\mathcal{D},\varepsilon}\!\left[ 2(f-\bar{\hat{f}})(\bar{\hat{f}}-\hat{f}) \right] = 2(f-\bar{\hat{f}})\,\mathbb{E}_{\mathcal{D}}\!\left[ \bar{\hat{f}}-\hat{f} \right] = 0 $$ Este término no contiene $\varepsilon$, por lo que la esperanza actúa únicamente sobre $\mathcal{D}$. La parte constante $(f-\bar{\hat{f}})$ sale fuera. Por la propia definición del modelo medio en <a href="#eq-modelo-medio" class="quarto-xref">Ecuación 4</a>, la esperanza de las desviaciones de un modelo respecto a su propia media es cero: $\mathbb{E}_{\mathcal{D}}\!\left[ \bar{\hat{f}}-\hat{f} \right] = \bar{\hat{f}} - \bar{\hat{f}} = 0$.

Al anularse todos los cruces, solo sobreviven las esperanzas de los tres cuadrados individuales:

$$
\mathbb{E}_{\varepsilon}\!\left[ \varepsilon^2 \right] + (f-\bar{\hat{f}})^2 + \mathbb{E}_{\mathcal{D}}\!\left[ (\bar{\hat{f}}-\hat{f})^2 \right]
$$

Como el ruido tiene media cero, sabemos que $\mathbb{E}_{\varepsilon}\!\left[ \varepsilon^2 \right] = \mathrm{Var}\!\left( \varepsilon \right) = \sigma^2$. Sustituyendo esto, obtenemos exactamente la identidad de <a href="#eq-descomposicion" class="quarto-xref">Ecuación 5</a>.

Estos tres sumandos no solo explican por qué nos equivocamos, sino que nos dan el diagnóstico exacto de **cómo de bien va a funcionar nuestro procedimiento de aprendizaje** ante datos futuros. Nos dicen si nuestras predicciones fallarán porque somos demasiado rígidos (sesgo), demasiado inestables (varianza) o porque la naturaleza es inherentemente impredecible (ruido):

- **El ruido** ($\sigma^2$) es la varianza intrínseca de $Y$ una vez fijado $\mathbf{x}$. Ningún procedimiento, por muchos datos que tenga o muy avanzado que sea el algoritmo, bajará de aquí: es el suelo inexpugnable del error que el capítulo 2 anunció.

- **El sesgo al cuadrado** mide la distancia entre la señal verdadera $f$ y nuestro modelo medio $\bar{\hat{f}}$ (la separación entre la curva negra y la curva gruesa naranja de <a href="#fig-variabilidad" class="quarto-xref">Figura 5</a>). Este error **no se atenúa con más datos**, ya que un procedimiento demasiado rígido (como ajustar líneas rectas) nunca podrá adoptar una forma curva por muchas observaciones que reciba. Depende exclusivamente de la clase de hipótesis elegida. **La única forma de reducir el sesgo es aumentar la complejidad** del modelo (por ejemplo, subiendo el grado del polinomio o añadiendo variables), dotando a nuestro procedimiento de la capacidad geométrica necesaria para calcar la señal real.

- **La varianza** mide cuánto se mueve un modelo concreto $\hat{f}_{\mathcal{D}}$ alrededor de su propio promedio $\bar{\hat{f}}$ por culpa de la muestra específica que le tocó (la dispersión de las finas líneas naranjas). Es el precio directo de la flexibilidad: al aumentar la complejidad para reducir el sesgo, le damos al algoritmo tanta libertad que empieza a perseguir el ruido de cada muestra. Sin embargo, a diferencia del sesgo, la varianza **sí se atenúa con más datos**, ya que un conjunto de entrenamiento más grande estabiliza el algoritmo y evita que el modelo dé bandazos.

La ecuación anterior detalla el error para un solo punto fijo $\mathbf{x}$. Para calcular el riesgo esperado de **?@def-riesgo-procedimiento**, solo nos falta un último paso: promediar ese resultado sobre todas las posibles características $\mathbf{x}$ que nos encontraremos en la realidad (es decir, tomar la esperanza sobre la distribución marginal de $X$ bajo $P^\star$).

Como la esperanza es un operador lineal, la descomposición sobrevive intacta a nivel global:

$$
\bar{R}_{n}=\mathbb{E}_{\mathcal{D}}\!\left[ R(\hat{f}_{\mathcal{D}}) \right]
=\sigma^2
+\mathbb{E}_{X}\!\left[ \bigl(f(X)-\bar{\hat{f}}(X)\bigr)^2 \right]
+\mathbb{E}_{X}\!\left[ \mathbb{E}_{\mathcal{D}}\!\left[ \bigl(\hat{f}_{\mathcal{D}}(X)-\bar{\hat{f}}(X)\bigr)^2 \right] \right] .
 \qquad(6)$$

Ahora sí, podemos medir de forma rigurosa estos tres componentes sobre el experimento simulado de la sección anterior.

In [ ]:
# TODO: completar en clase

La última columna es la suma de los tres términos, y <a href="#eq-descomposicion" class="quarto-xref">Ecuación 5</a> dice que esa suma es el riesgo esperado. Conviene comprobarlo midiendo el riesgo esperado por el otro camino: el promedio, sobre los dos mil ajustes, del error cuadrático que cada uno comete contra la señal.

In [ ]:
# TODO: completar en clase

La tabla revela la tensión fundamental del aprendizaje automático:

- El sesgo al cuadrado cae drásticamente desde $0.4988$ (grado 0) hasta $0.0046$ al llegar al grado 3. A partir de ahí, apenas varía. La razón geométrica es simple: un polinomio cúbico ya es lo suficientemente flexible para imitar la forma de la onda $\sin(\pi x)$ en este intervalo. Añadir potencias mayores no acerca mucho más el modelo medio a la señal real porque el problema de “falta de capacidad” ya está resuelto.

- **La varianza explota:** La varianza hace exactamente lo contrario. Se mantiene baja hasta el grado 3 ($0.0066$) y luego se descontrola subiendo hasta $3.1114$ para grado 9. El motivo es la sobreparametrización: al intentar estimar 10 parámetros con solo 36 observaciones repartidas al azar, el modelo tiene demasiada libertad. El polinomio persigue el ruido específico de cada muestra y se dispara en los huecos entre los puntos, provocando que la curva cambie radicalmente de una muestra a otra.

- **El riesgo total sigue una curva en U:** La suma de los tres términos (sesgo, varianza y el ruido constante de $0.0400$) es el riesgo esperado. Al chocar la caída del sesgo con la explosión de la varianza, el riesgo total forma una U. El mínimo no está en los extremos, sino en el interior: el grado 3 encuentra el equilibrio óptimo.

## La curva en U y lo que la validación cruzada logra ver

La suma de la sección anterior (el riesgo esperado total) solo se puede calcular porque en esta simulación conocemos la señal verdadera y podemos promediar miles de muestras. En el mundo real, solo tenemos una única muestra y la señal es desconocida.

¿Cómo estimamos entonces esta curva en la práctica? Recordando lo que establecimos en **?@sec-kfold**: la validación cruzada no evalúa el modelo final, sino que **estima el riesgo esperado del procedimiento** de **?@def-riesgo-procedimiento**. Al partir nuestra única muestra en bloques y ajustar múltiples modelos, la cantidad de **?@eq-cv** simula empíricamente esa esperanza $\mathbb{E}_{\mathcal{D}}\!\left[ \cdot \right]$ teórica. Conviene comprobar qué es lo que realmente logra ver frente a la verdad teórica.

In [ ]:
# TODO: completar en clase

Figura 6: A la izquierda, la descomposición: la rama izquierda de la curva del riesgo es sesgo y la derecha es varianza. A la derecha, lo que se puede calcular con una sola muestra de treinta y seis observaciones. La validación cruzada reproduce la rama izquierda con nitidez y la derecha solo a partir del grado 9.

In [ ]:
grados = list(range(10))

fig, ax = plt.subplots(1, 2, figsize=(8.4, 3.2))

ax[0].plot(grados, [descomposicion[g][0] for g in grados], marker="o", ms=4,
           color=AZUL, label="sesgo$^2$")
ax[0].plot(grados, [descomposicion[g][1] for g in grados], marker="s", ms=4,
           color=NARANJA, label="varianza")
ax[0].plot(grados, [descomposicion[g][2] for g in grados], marker="^", ms=4,
           color=TINTA, label="riesgo esperado")
ax[0].axhline(NOISE ** 2, color=GRIS, lw=1, linestyle=":",
              label=r"ruido $\sigma^2$")
ax[0].set(xlabel="grado del polinomio", ylabel="riesgo cuadrático medio",
          yscale="log", title="con la señal conocida")
ax[0].legend(fontsize=8)

ax[1].plot(grados, [tabla_cv[g] for g in grados], marker="o", ms=4,
           color=NARANJA, label="CV(5)")
ax[1].plot(grados, [descomposicion[g][2] for g in grados], marker="^", ms=4,
           color=TINTA, label="riesgo esperado")
ax[1].axhline(NOISE ** 2, color=GRIS, lw=1, linestyle=":")
ax[1].set(xlabel="grado del polinomio", yscale="log",
          title="con una sola muestra")
ax[1].legend(fontsize=8)

plt.tight_layout()

La comparación de los dos paneles nos advierte de los límites de medir con muestras pequeñas. La validación cruzada ve la rama del sesgo perfectamente: distingue sin ambigüedad los grados 0, 1 y 2 del resto. Sin embargo, en la rama de la varianza fracasa: entre los grados 3 y 8 devuelve valores estancados entre $0.0286$ y $0.0323$, colocando su aparente mínimo en el grado 8.

Aquí debemos separar rigurosamente dos cantidades que conviene no confundir:

1.  El error de estimación: La validación cruzada nos promete que el riesgo del grado 8 es $0.0286$, pero la verdad teórica es $0.4239$. La herramienta es demasiado optimista y se equivoca por un factor de quince al evaluar este candidato.

2.  El coste de elegir mal: Si caemos en la trampa del mínimo aparente y entregamos el grado 8, el riesgo real será $0.4239$. Si entregamos el grado 3, el riesgo será de $0.0512$. Elegir mal multiplica nuestro error en producción por ocho.

¿Por qué se equivoca la CV? Es **?@prp-minimo-sesgado** otra vez: el mínimo de varias estimaciones ruidosas es optimista y cae donde el ruido lo lleve. La regla de un error típico (**?@def-regla-un-error-tipico**) no mejora esa estimación, pero sí cambia la decisión. Apliquémosla sobre esta tabla, con sus tres pasos.

In [ ]:
# TODO: completar en clase

El campeón es el grado 8, y cuatro candidatos más simples quedan a menos de un error típico de él. La regla se queda con el más simple de los cuatro, el grado 3, con un cociente de $0.48$, y ese es también el que minimiza el riesgo esperado, $0.0512$: recupera el mejor candidato y evita los $0.4239$ del campeón.

Conviene no leer eso como una garantía. La regla de un error típico es una heurística, y no hay ningún resultado que asegure que el candidato que devuelve minimice el riesgo esperado. El último bloque de la salida lo mide sobre estas mismas treinta y seis observaciones, cambiando solo la partición en bloques: en 27 de las 40 particiones la regla entrega el grado 3, en 10 el grado 4, cuyo riesgo esperado es $0.0563$ frente a $0.0512$, y en las 3 restantes un grado más alto.

Lo que la regla cambia es la pregunta que decide. El mínimo de la curva de validación pregunta cuál es el número más bajo, y por **?@prp-minimo-sesgado** ese número cae donde lo lleve el ruido. La regla pregunta cuál es el candidato más simple cuya desventaja frente al campeón no se “distingue del ruido”, y esa pregunta se contesta con la diferencia emparejada de **?@def-diferencia-emparejada**, que cancela la dificultad común de los bloques y mide mucho mejor la comparación que la propia columna `ee`.

**El doble descenso queda fuera.** Con clases de hipótesis muy grandes, del orden del número de observaciones o mayores, la curva en U deja de ser una U: el riesgo vuelve a bajar pasado el punto en que el modelo interpola los datos. El fenómeno se llama doble descenso, es importante para entender modelos con millones de parámetros y no es evaluable en este curso, porque exige herramientas que no vamos a ver.

## Caso práctico: la curva en U en los datos de Madrid

Volvamos a los anuncios del capítulo 5, con el mismo reparto por anfitrión y el mismo protocolo. El punto de partida es el modelo que aquel capítulo entregó: recorte al percentil 99, imputación por la mediana con indicadora, estandarización, tipo de habitación y barrio agrupado, y un $\mathrm{RMSE}$ de $82.39$ euros en test. La pregunta que intentaremos resolver es si una clase de hipótesis más rica lo mejora.

Vamos a subir el grado del polinomio y estudiar qué le sucede al riesgo. Las variables que se van a aumentar son las cuatro continuas de tamaño, `accommodates`, `bathrooms`, `bedrooms` y `beds`, y lo único que cambia de un candidato a otro es el grado.

In [ ]:
import warnings

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import MissingIndicator, SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import (OneHotEncoder, PolynomialFeatures,
                                   StandardScaler)

warnings.simplefilter("ignore")

anuncios = pd.read_csv("../datos/airbnb_madrid.csv")

NUMERICAS = ["accommodates", "bathrooms", "bedrooms", "beds",
             "minimum_nights", "number_of_reviews",
             "review_scores_rating", "availability_365"]
CATEGORICAS = ["room_type", "neighbourhood_cleansed"]
TAMANO = ["accommodates", "bathrooms", "bedrooms", "beds"]
RESTO = [c for c in NUMERICAS if c not in TAMANO]

indices_train, indices_test = next(
    GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=0)
    .split(anuncios, groups=anuncios["host_id"]))
train_a = anuncios.iloc[indices_train]
test_a = anuncios.iloc[indices_test]
y_train_a = train_a["price"].to_numpy()
y_test_a = test_a["price"].to_numpy()
grupos_a = train_a["host_id"].to_numpy()


def mse(y_pred, y):
    return float(np.mean((np.asarray(y_pred) - np.asarray(y)) ** 2))


print(f"train {len(train_a)} anuncios, test {len(test_a)} anuncios")

In [ ]:
# TODO: completar en clase

Para medir el efecto de la complejidad sin cambiar nada más, una única función construye el *pipeline* completo a partir del grado que se le pide.

In [ ]:
# TODO: completar en clase

La tabla dice tres cosas.

1.  **Algo de flexibilidad baja el sesgo.** El grado 2 añade diez columnas, cuatro cuadrados y seis productos entre pares de variables de tamaño, que son las interacciones de la sección segunda. El riesgo de validación baja $192.1\pm65.5$, casi tres errores típicos: sobre $7096.7$ es una mejora del 3 %, y dice que el modelo lineal del capítulo 5 tenía algo de sesgo por resolver.
2.  **Más flexibilidad se paga en la precisión de la medición.** El grado 4 necesita 153 columnas y deja el riesgo en $7927.6$, con una diferencia de $+830.9\pm947.1$. El error típico emparejado se ha multiplicado por catorce respecto de la fila anterior, y con cinco bloques eso deja la comparación sin veredicto: no se puede afirmar que empeore, y tampoco hay ningún indicio de que mejore. La rama derecha de la curva en U no aparece aquí como un riesgo alto y claro, sino como una estimación que se vuelve demasiado imprecisa para decidir.
3.  **La preparación de los datos y la clase de hipótesis no son decisiones independientes.** El mismo grado 2, sin el recorte al percentil 99, deja el riesgo en $10\,229.9$, un 44 % peor que la referencia. Elevar al cuadrado una columna que llega a 167 camas produce valores de otro orden de magnitud, y el ajuste se va detrás de ellos. Su error típico emparejado, $1823.8$, es el mayor de la tabla: el candidato no es peor de forma estable, es inestable de un bloque a otro.

In [ ]:
# TODO: completar en clase

La validación cruzada elige el grado 2, y el test se abre una vez para medirlo. En esa misma apertura se mide también el modelo del capítulo 5, porque las dos predicciones caen sobre los mismos 3784 anuncios y eso permite comparar emparejando; ninguno de los dos candidatos se ha elegido mirando el test.

El elegido comete un $\mathrm{RMSE}$ de $80.68$ euros frente a los $82.39$ del capítulo 5. Son $1.7$ euros sobre una mediana de 124. Los dos modelos se evalúan sobre los mismos 3784 anuncios, así que se emparejan igual que en **?@sec-comparar**, con la observación en lugar del bloque como unidad: la diferencia emparejada es $-279.6\pm64.5$, más de cuatro errores típicos. El error típico del riesgo de cada modelo por separado, unos $570$, no es el que corresponde. La mejora es pequeña y es real.

El balance del capítulo es el siguiente. Sabemos construir clases de hipótesis tan flexibles como queramos; hemos demostrado que el error se descompone en ruido, sesgo y varianza; y hemos comprobado cómo la varianza se dispara con la complejidad hasta arruinar el modelo. También hemos aprendido la regla de oro: más datos curan la varianza, pero no el sesgo. Por tanto, ante un resultado insuficiente, nuestro primer paso será siempre diagnosticar en qué rama de la curva en U nos encontramos.

Lo que aún nos falta es un mecanismo para conservar la flexibilidad sin pagar el altísimo precio de la varianza. Hasta ahora, nuestra única forma de estabilizar el algoritmo ha sido amputar características; es decir, retroceder a una clase de hipótesis más pequeña y rígida. El próximo capítulo introduce la gran alternativa a esta renuncia.

## Ejercicios

<span class="theorem-title">**Ejercicio 1 (Otra base de características)**</span>  

Las potencias no son la única forma de construir características. Otra es colocar gaussianas sobre el recorrido de $x$,

$$
\varphi_j(x)=\exp\Bigl(-\frac{(x-c_j)^2}{2h^2}\Bigr),
$$

con centros $c_1,\ldots,c_{d}$ repartidos por él y una anchura $h$ común.

1.  Construye la matriz de características con ocho centros equiespaciados en $[0,2]$ y $h=0.25$, ajusta el modelo sobre las treinta y seis observaciones del capítulo y dibuja la curva junto a la señal.
2.  Repite con $h=0.6$ y di cuál de los dos términos de <a href="#thm-descomposicion" class="quarto-xref">Teorema 1</a> que dependen del modelo crece. Repite después con $h=0.05$ y explica por qué con treinta y seis observaciones la descomposición no se puede leer; rehazla con $n=300$.
3.  Razona por qué $h$ es un parámetro de complejidad aunque el número de características no cambie.

<span class="theorem-title">**Ejercicio 2 (Comprobar la descomposición)**</span> El capítulo calcula los tres términos de <a href="#eq-descomposicion" class="quarto-xref">Ecuación 5</a> por separado y afirma que suman el riesgo esperado.

1.  <span class="nuevo">El capítulo comprueba la identidad con las mismas dos mil muestras en los dos caminos, y por eso los dos números coinciden hasta el último decimal. Explica por qué esa coincidencia es algebraica y no estadística, y qué comprobación haría falta para que fuera estadística.</span>
2.  Repite <span class="nuevo">la tabla</span> con doscientas repeticiones en lugar de dos mil y explica qué término se estima peor. <span class="nuevo">Estima la varianza del grado 9 con tres semillas distintas y di hasta qué cifra se puede leer.</span>
3.  Comprueba que el término de ruido no cambia al variar el grado, y di qué línea del código lo garantiza por construcción.

<span class="theorem-title">**Ejercicio 3 (Cuántas observaciones necesita la validación cruzada)**</span> Con una muestra de <span class="nuevo">treinta y seis</span> observaciones, la validación cruzada del capítulo no distingue los grados 3 a 8 y su mínimo cae en el <span class="nuevo">8</span>.

1.  Repite la tabla de validación cruzada con $n=100$ y con $n=400$ y anota en cada caso el grado del mínimo.
2.  Compara en cada caso el error típico entre bloques con la diferencia entre el grado 3 y el grado 7 del riesgo esperado. Explica la relación con lo que observas en a.
3.  Repite la tabla con $n=30$ y diez semillas distintas, y cuenta cuántas veces el mínimo cae en cada grado. Relaciona la dispersión con **?@prp-minimo-sesgado**.

<span class="theorem-title">**Ejercicio 4 (Elegir interacciones con un motivo)**</span>  

El grado 2 del caso práctico cruza las variables de tamaño entre sí, pero no con las categóricas. La interacción de <a href="#fig-interaccion" class="quarto-xref">Figura 4</a>, la que hace que una plaza más valga distinto en una vivienda completa y en una habitación privada, no está en ninguno de los cuatro candidatos.

1.  Añade un candidato que multiplique las cuatro variables de tamaño por las indicadoras del tipo de habitación, y mide su estimación por validación cruzada. El cruce tiene que ser un transformador dentro del *pipeline*, no una columna añadida a la tabla, para que sus medianas y sus percentiles se estimen en cada bloque: es la regla del capítulo 5.
2.  Añade otro que cruce con el distrito en lugar de con el tipo de habitación, cuenta cuántas columnas produce y explica el resultado con la descomposición.
3.  Un compañero propone probar todos los cruces posibles de una variable numérica por una categórica y quedarse con el mejor. Di qué objeción tiene ese plan y qué habría que hacer en su lugar.